### Continuum intensity in LTE

Some imports from Project 2.

In [4]:
%matplotlib inline
# Only if you have a high-resolution "retina" display:
%config InlineBackend.figure_format = 'retina'

import numpy
import matplotlib.pyplot as plt

from astropy import units
from astropy import constants
from astropy.io import fits
from astropy.table import QTable  # to use tables with units
from astropy.modeling.models import BlackBody
from astropy.visualization import quantity_support

from scipy.integrate import cumulative_trapezoid
from scipy.ndimage import shift  # for "rotating" 3D cubes


quantity_support()
plt.rc('legend', frameon=False)
plt.rc('figure', figsize=(7, 7 / 1.75)) # Larger figure sizes
plt.rc('font', size=12)

i_units = units.Quantity(1, unit="kW m-2 sr-1 nm-1")  # More practical SI units

In [5]:
def cross_section_hminus_bf(wavelength, temperature):
    """
    Gets the H^- bound-free cross section using the recipes of
    Wishart (1979) and Broad and Reinhardt (1976).
    
    Parameters
    ----------
    wavelength : astropy.units.quantity (scalar or array)
        Wavelength to calculate in units of length.
    temperature: astropy.units.quantity (scalar or array)
        Gas temperature in units of K or equivalent.
        
    Returns
    -------
    cross_section : astropy.units.quantity (scalar or array)
        cross section (in m^2) per H^- ion.
    """
    wbr_λ = numpy.array(
        [  18, 19.6, 21.4, 23.6, 26.4, 29.8, 34.3, 40.4, 49.1, 62.6,  121, 139,
          164,  175,  200,  225,  250,  275,  300,  325,  350,  375,  400, 425,
          450,  475,  500,  525,  550,  575,  600,  625,  650,  675,  700, 725,
          750,  775,  800,  825,  850,  875,  900,  925,  950,  975, 1000, 1025,
         1050, 1075, 1100, 1125, 1150, 1175, 1200, 1225, 1250, 1275, 1300, 1325,
         1350, 1375, 1400, 1425, 1450, 1475, 1500, 1525, 1550, 1575, 1600, 1610,
         1620, 1630]
    ) * units.nm  # in nm
    wbr_σ = numpy.array(
        [0.067, 0.088, 0.117, 0.155, 0.206, 0.283, 0.414, 0.703,  1.24,  2.33,
          5.43,  5.91,  7.29, 7.918, 9.453, 11.08, 12.75, 14.46, 16.19, 17.92,
         19.65, 21.35, 23.02, 24.65, 26.24, 27.77, 29.23, 30.62, 31.94, 33.17,
         34.32, 35.37, 36.32, 37.17, 37.91, 38.54, 39.07, 39.48, 39.77, 39.95,
         40.01, 39.95, 39.77, 39.48, 39.06, 38.53, 37.89, 37.13, 36.25, 35.28,
         34.19, 33.01, 31.72, 30.34, 28.87, 27.33, 25.71, 24.02, 22.26, 20.46,
         18.62, 16.74, 14.85, 12.95, 11.07, 9.211, 7.407, 5.677, 4.052, 2.575,
         1.302, 0.8697, 0.4974, 0.1989]
    ) * 1e-22 * units.m**2
    sigma = numpy.interp(wavelength, wbr_λ, wbr_σ)
    # correct for stimulated emission
    sigma = sigma *  (1 - numpy.exp(-constants.h * constants.c /
                                    (wavelength * constants.k_B * temperature)))
    return sigma
    

def cross_section_hminus_ff(wavelength, temperature):
    """
    Gets the H^- free-free cross section coefficient using the recipe
    of John (1988). Includes stimulated emission.
    
    Parameters
    ----------
    wavelength : astropy.units.quantity (scalar)
        Wavelength to calculate in units of length.
    temperature: astropy.units.quantity (scalar or array)
        Gas temperature in units of K or equivalent.
        
    Returns
    -------
    cross_section : astropy.units.quantity (scalar or array)
        H^- ff cross section coefficient (in m^5) per neutral hydrogen atom per electron.
    """
    table = numpy.array([
        [    0.0000,     0.0000,      0.0000,      0.0000,     0.0000,    0.0000],
        [ 2483.3460,   285.8270,  -2054.2910,   2827.7760, -1341.5370,  208.9520],
        [-3449.8890, -1158.3820,   8746.5230, -11485.6320,  5303.6090, -812.9390],
        [ 2200.0400,  2427.7190, -13651.1050,  16755.5240, -7510.4940, 1132.7380],
        [ -696.2710, -1841.4000,   8624.9700, -10051.5300,  4400.0670, -655.0200],
        [   88.2830,   444.5170,  -1863.8640,   2095.2880,  -901.7880,  132.9850]]
    ) 
    sqrtθ = numpy.sqrt(5040 / temperature.to_value("K"))
    wave_mu = wavelength.to_value("um")
    wave_inv = 1. / wave_mu
    kappa = 0.
    for i in range(6):
        kappa += sqrtθ**(i + 2) * (wave_mu**2 * table[i, 0] + table[i, 1] +
                       wave_inv * (table[i, 2] + wave_inv * (table[i, 3] +
                       wave_inv * (table[i, 4] + wave_inv * table[i, 5]))))
    kappa = kappa * 1e-32 * units.m**4 / units.N  # Put units from table
    return (kappa * constants.k_B * temperature).to("m^5")  

In [6]:
def compute_h_neutral_frac(temperature, electron_density):
    """
    Computes the fraction of neutral hydrogen for a given temperature
    and electron density.
    """
    chi_H = 2.1787094174620437 * units.aJ
    saha_const = ((2 * numpy.pi * constants.m_e * constants.k_B) / constants.h**2)**1.5
    saha = (saha_const * temperature**1.5 / electron_density * 
            numpy.exp(-chi_H / (constants.k_B * temperature)))
    return 1 / (1 + saha)

def compute_hminus_frac(temperature, electron_density):
    """
    Computes the fraction of H- divided by neutral hydrogen for a 
    given temperature and electron density.
    """
    chi_Hminus = 0.12080412 * units.aJ
    saha_const = ((2 * numpy.pi * constants.m_e * constants.k_B) / constants.h**2)**1.5
    saha = (4 * saha_const * temperature**1.5 / electron_density * 
            numpy.exp(-chi_Hminus / (constants.k_B * temperature)))
    return 1 / saha

def compute_α_cont(wavelength, temperature, electron_density,
                   h_density, h_minus=None):
    """
    Computes continuum extinction coefficient α using
    Thomson scattering plus the H- bf and ff extinction.
    """
    h_neutral = h_density * compute_h_neutral_frac(temperature, electron_density)
    if h_minus is None:
        # If not provided, compute from neutral hydrogen
        h_minus = h_neutral * compute_hminus_frac(temperature, electron_density)
    alpha_bf = cross_section_hminus_bf(wavelength, temperature) * h_minus
    alpha_ff = cross_section_hminus_ff(wavelength, temperature) * h_neutral * electron_density
    alpha_thomson = 6.648e-29 * units.m ** 2 * electron_density
    return alpha_bf + alpha_ff + alpha_thomson

In [7]:
def lte_intensity(wavelength, distance, temperature, extinction, mu=1):
    """
    Solves the radiative transfer equation assuming LTE for a single ray.
    
    Parameters
    ----------
    wavelength: astropy.units.quantity (scalar)
        Wavelength to calculate, in units of length.
    distance : astropy.units.quantity (1-D array)
        Distances along path of ray, in units of length. Can be different
        length than wavelength array.
    temperature: astropy.units.quantity (n-D array)
        Gas temperature in units of K or equivalent, for all points along
        the ray. Same length as distance.
    extinction: astropy.units.quantity (n-D array)
        Extinction coefficient in units of inverse length, for all
        points along the ray. Same dimensions as temperature.
    """
    tau = cumulative_trapezoid(extinction, x=distance, initial=0, axis=0) / mu
    source_function = BlackBody(temperature, scale=i_units)(wavelength)
    return numpy.trapz(source_function * numpy.exp(-tau), tau, axis=0)

Now loading the 3D model:

In [8]:
# For jupyterhub.uio.no:
DATA_FILE = "/src/data/AST4310/qs006024_sap_s285.fits"
# For local file in your computer
DATA_FILE = "qs006024_sap_s285.fits"
atm3d = QTable.read(DATA_FILE)
atm3d

temperature,electron_density,hydrogen_density,velocity_z,height,velocity_y,velocity_x,pressure
K,1 / m3,1 / m3,m / s,m,m / s,m / s,Pa
"float32[256,256]","float32[256,256]","float32[256,256]","float32[256,256]",float64,"float32[256,256]","float32[256,256]","float32[256,256]"
5272.96923828125 .. 5284.5224609375,4.0828113734598656e+17 .. 4.22937661703979e+17,1.1546960635975193e+21 .. 1.1705831446017663e+21,157.57679748535156 .. 188.99781799316406,583024.0,-591.6886596679688 .. -594.4967041015625,151.15953063964844 .. 138.1940155029297,92.58880615234375 .. 94.07086181640625
5403.09130859375 .. 5411.1669921875,5.892596861260268e+17 .. 6.056246110354145e+17,1.268329059807983e+21 .. 1.2855685577065806e+21,279.1968994140625 .. 312.5419616699219,570330.6,-607.3775634765625 .. -610.0527954101562,148.22718811035156 .. 134.47979736328125,104.22808837890625 .. 105.80181884765625
5457.4619140625 .. 5460.5078125,7.130042222740767e+17 .. 7.238788733091185e+17,1.4060311220144633e+21 .. 1.4257131190234676e+21,397.9075622558594 .. 431.21435546875,557637.2000000001,-673.2451171875 .. -675.8231811523438,133.34869384765625 .. 116.03425598144531,116.7034912109375 .. 118.40267181396484
5481.47607421875 .. 5478.60009765625,8.02345177027969e+17 .. 8.032026586586808e+17,1.5645948428573422e+21 .. 1.5871253666306583e+21,493.31866455078125 .. 525.1014404296875,544943.7999999999,-847.243896484375 .. -849.96142578125,92.8567886352539 .. 66.19844055175781,130.43585205078125 .. 132.2436981201172
5437.236328125 .. 5425.58447265625,7.704930810855424e+17 .. 7.55770551670145e+17,1.7582095166500923e+21 .. 1.7849445728880238e+21,576.6238403320312 .. 605.1107177734375,532250.4,-1198.0218505859375 .. -1198.6129150390625,34.679115295410156 .. -6.235204219818115,145.38577270507812 .. 147.28074645996094
5307.6630859375 .. 5274.84521484375,6.150925743816704e+17 .. 5.789578806490563e+17,2.0056081475177378e+21 .. 2.0432271374176286e+21,639.0731811523438 .. 667.4573364257812,519557.00000000006,-1625.5244140625 .. -1612.75390625,-4.219546318054199 .. -54.43379211425781,161.87940979003906 .. 163.8760986328125
...,...,...,...,...,...,...,...
11589.94921875 .. 11593.806640625,2.6216174006864036e+22 .. 2.6315467119648488e+22,2.038237317915488e+23 .. 2.0436311891171972e+23,2663.761962890625 .. 2643.78955078125,-302254.7,361.02813720703125 .. 359.9884338378906,728.10498046875 .. 742.7396850585938,40124.42578125 .. 40248.75


### Some exercises
* Compute the continuum intensity from the 3D model at $\mu=1$
* Compute the contribution function for a few columns
* Compute the continuum intensity from the 3D model at $\mu=0.4$